In [ ]:
# === Cell 1: Data Root ===
import os

# Data root; override with the KI_SORT_ROOT environment variable.
KI_SORT_ROOT = os.environ.get("KI_SORT_ROOT", "/VOL_D/fio.h")


In [ ]:
# === Cell 2: Prerequisites ===
# Conda environment ir-class for IR classification, registered as a Jupyter kernel.
"""
conda create -n ir-class python=3.10 -y
conda activate ir-class

conda install -y -c conda-forge \
  ipykernel numpy pandas matplotlib seaborn scikit-learn

pip install --no-cache-dir joblib

python -m ipykernel install --user --name ir-class --display-name "ir-class"
"""

In [ ]:
# === Cell 3: Imports ===

import os
import re
import sys
import json
import random
import platform
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline        import Pipeline
from sklearn.preprocessing   import StandardScaler
from sklearn.decomposition   import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.svm             import SVC
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.ensemble        import RandomForestClassifier
from sklearn.tree            import DecisionTreeClassifier
from sklearn.linear_model    import LogisticRegression
from sklearn.base            import clone
from sklearn.metrics         import (
    f1_score, balanced_accuracy_score, precision_score,
    recall_score, accuracy_score, confusion_matrix,
    roc_auc_score,
)
from sklearn.preprocessing   import label_binarize
import joblib

from IPython.display import display

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"]      = 120
plt.rcParams["savefig.dpi"]     = 150
plt.rcParams["figure.figsize"]  = (8, 5)
plt.rcParams["axes.titlesize"]  = 12
plt.rcParams["axes.labelsize"]  = 11
plt.rcParams["legend.fontsize"] = 10
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10

In [ ]:
# === Cell 4: Config ===

# --- Input paths ---
SPECTRA_DIR      = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Balanced_NIR_Dataset/01_Preprocessing_Investigation/C2_als_z-score")
OBJECT_SPLIT_CSV = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/01_LS_Export/20260518_Export/object_split.csv")
GROUND_TRUTH_CSV = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Balanced_NIR_Dataset/spectra_ground_truth_v7.csv")

# --- Output paths ---
BENCHMARK_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/05_IR_Classification/20260629_Classifier_Benchmark_Best_Preprocessing")
RESULTS_DIR    = BENCHMARK_ROOT / "results"
MODELS_DIR     = RESULTS_DIR / "models"
for p in [BENCHMARK_ROOT, RESULTS_DIR, MODELS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

PROJECT_NAME = BENCHMARK_ROOT.name
RANDOM_SEED  = 42

# Set to True to allow re-running into an existing results directory
# (checked in the Sanity Check cell).
ALLOW_OVERWRITE = False

# --- Dataset semantics ---
# Spectrum filename: <PREFIX_MATERIAL>_obj_<OBJECT_ID>_<REGION> (no extension).
# The prefix is only a naming convention; the true class of each region comes
# from the ground-truth CSV (e.g. an HDPE bottle can have a Paper label).
FILENAME_REGEX = re.compile(
    r"^(?P<material>[A-Za-z0-9]+)_obj_(?P<object_id>\d+)_(?P<region>body|cap|label)$",
    re.IGNORECASE,
)
OBJECT_ID_WIDTH = 2                         # zero-pad object ids to this width
VALID_REGIONS   = ["body", "cap", "label"]
VALID_SPLITS    = {"cv", "test"}            # "skipped" objects are excluded

# Ground-truth column holding the true class for each region.
REGION_TO_GT_COLUMN = {
    "body":  "body_class",
    "label": "label_class",
    "cap":   "cap_class",
}

# --- Outer split: objects not in the segmentation split ---
# Objects missing from object_split.csv go to the CV pool, except those listed
# here, which go to the test set (~21 % of objects, as in the segmentation split).
NEW_TEST_OBJECTS = [
    ("HDPE", 13),
    ("PET",  16), ("PET", 21), ("PET", 24), ("PET", 28),
    ("PS",   14), ("PS",  18),
]

# --- Cross-validation ---
NUM_FOLDS = 4
N_JOBS    = -1   # CPU cores for GridSearchCV

# --- Evaluation metrics ---
# macro-F1 is primary: it weights all polymer classes equally, including rare ones.
PRIMARY_METRIC    = "macro-F1"
SECONDARY_METRICS = ["balanced_accuracy", "macro-precision", "macro-recall", "accuracy"]

# --- PCA dimensionality search space ---
# Pipeline: scaler -> pca -> clf; PCA is fit on each fold's training spectra only.
# None keeps all components (min(n_train, n_features)). Values must stay below
# the per-fold training-sample count.
PCA_N_COMPONENTS_GRID = [5, 10, 20, 30, None]

# --- Classifier hyperparameter grids ---
# Each entry: (estimator, param_grid) with Pipeline prefix "clf__";
# "pca__n_components" is added to every grid below. Grids are saved to RESULTS_DIR.
# Spectra are already z-score normalised per sample by the preprocessing.

_CLASSIFIER_GRIDS = {
    "SVM": (
        SVC(probability=True, random_state=RANDOM_SEED, max_iter=10_000),
        [
            {"clf__kernel": ["linear"], "clf__C": [0.1, 1, 10, 100]},
            {"clf__kernel": ["rbf"],    "clf__C": [0.1, 1, 10, 100],
             "clf__gamma":  ["scale", 0.001, 0.01, 0.1]},
        ],
    ),
    "kNN": (
        KNeighborsClassifier(algorithm="auto"),
        [
            {"clf__n_neighbors": [1, 3, 5, 7, 9], "clf__weights": ["uniform", "distance"],
             "clf__metric": ["euclidean"]},
            {"clf__n_neighbors": [1, 3, 5, 7, 9], "clf__weights": ["uniform", "distance"],
             "clf__metric": ["cosine"], "clf__algorithm": ["brute"]},
        ],
    ),
    "RF": (
        RandomForestClassifier(random_state=RANDOM_SEED),
        {"clf__n_estimators": [200, 500], "clf__max_depth": [None, 5, 10, 20],
         "clf__min_samples_leaf": [1, 2, 5]},
    ),
    "DT": (
        DecisionTreeClassifier(criterion="gini", random_state=RANDOM_SEED),
        {"clf__max_depth": [None, 3, 5, 10, 20], "clf__min_samples_leaf": [1, 2, 5]},
    ),
    "LR": (
        # penalty defaults to "l2"; multinomial loss is automatic for lbfgs/saga.
        # penalty= and multi_class= are not set (deprecated in sklearn 1.5, removed in 1.7).
        LogisticRegression(max_iter=5_000, random_state=RANDOM_SEED),
        {"clf__C": [0.01, 0.1, 1, 10, 100], "clf__solver": ["lbfgs", "saga"]},
    ),
}

def _inject_pca_grid(param_grid):
    """Add pca__n_components to every sub-grid so PCA is tuned with the classifier."""
    if isinstance(param_grid, list):
        return [{**sub, "pca__n_components": PCA_N_COMPONENTS_GRID} for sub in param_grid]
    return {**param_grid, "pca__n_components": PCA_N_COMPONENTS_GRID}

PARAM_GRIDS = {
    name: (est, _inject_pca_grid(grid))
    for name, (est, grid) in _CLASSIFIER_GRIDS.items()
}

# --- Output artefact paths ---
SAMPLE_TABLE_CSV     = RESULTS_DIR / f"{PROJECT_NAME}_sample_table.csv"
FOLD_ASSIGNMENT_CSV  = RESULTS_DIR / f"{PROJECT_NAME}_fold_assignments.csv"
FOLD_REASSIGN_CSV    = RESULTS_DIR / f"{PROJECT_NAME}_fold_reassignments.csv"
SPLIT_OBJECTS_CSV    = RESULTS_DIR / f"{PROJECT_NAME}_cv_split_objects.csv"
SPLIT_SUMMARY_CSV    = RESULTS_DIR / f"{PROJECT_NAME}_split_summary.csv"
HPO_RESULTS_CSV      = RESULTS_DIR / f"{PROJECT_NAME}_hpo_results.csv"
CV_METRICS_CSV       = RESULTS_DIR / f"{PROJECT_NAME}_cv_metrics.csv"
CV_PER_CLASS_CSV     = RESULTS_DIR / f"{PROJECT_NAME}_cv_per_class_metrics.csv"
TEST_METRICS_CSV     = RESULTS_DIR / f"{PROJECT_NAME}_test_metrics.csv"
TEST_PREDICTIONS_CSV = RESULTS_DIR / f"{PROJECT_NAME}_test_predictions.csv"
PARAM_GRIDS_JSON     = RESULTS_DIR / f"{PROJECT_NAME}_param_grids.json"
CONFUSION_MATRIX_PNG = RESULTS_DIR / f"{PROJECT_NAME}_confusion_matrices.png"
CV_COMPARISON_PNG    = RESULTS_DIR / f"{PROJECT_NAME}_cv_comparison.png"
TEST_COMPARISON_PNG  = RESULTS_DIR / f"{PROJECT_NAME}_test_comparison.png"
BRIER_SCORE_PNG      = RESULTS_DIR / f"{PROJECT_NAME}_brier_score.png"
AUC_ROC_PNG          = RESULTS_DIR / f"{PROJECT_NAME}_auc_roc.png"

# --- Sanity printout ---
print("Configuration")
print("-" * 80)
print("SPECTRA_DIR:         ", SPECTRA_DIR)
print("OBJECT_SPLIT_CSV:    ", OBJECT_SPLIT_CSV)
print("GROUND_TRUTH_CSV:    ", GROUND_TRUTH_CSV)
print("BENCHMARK_ROOT:      ", BENCHMARK_ROOT)
print("RESULTS_DIR:         ", RESULTS_DIR)
print("MODELS_DIR:          ", MODELS_DIR)
print("PROJECT_NAME:        ", PROJECT_NAME)
print("RANDOM_SEED:         ", RANDOM_SEED)
print("NUM_FOLDS:           ", NUM_FOLDS)
print("N_JOBS:              ", N_JOBS, "(CPU; sklearn classifiers have no GPU path)")
print("PRIMARY_METRIC:      ", PRIMARY_METRIC)
print("PCA_N_COMPONENTS_GRID:", PCA_N_COMPONENTS_GRID)
print("Classifiers:         ", list(PARAM_GRIDS.keys()))

In [ ]:
# === Cell 5: Sanity Check ===

# Stop if a previous run's artefacts exist and ALLOW_OVERWRITE is False.
_existing = [p for p in [CV_METRICS_CSV, TEST_METRICS_CSV] if p.exists()]
if _existing and not ALLOW_OVERWRITE:
    raise RuntimeError(
        "Existing benchmark artefacts found in RESULTS_DIR and ALLOW_OVERWRITE is "
        f"False:\n  " + "\n  ".join(str(p) for p in _existing) +
        "\nSet ALLOW_OVERWRITE = True in Cell 4 to re-run into this directory."
    )

import sklearn
print("Python:       ", sys.version.split()[0])
print("numpy:        ", np.__version__)
print("pandas:       ", pd.__version__)
print("scikit-learn: ", sklearn.__version__)

PATHS_TO_CHECK = [
    ("SPECTRA_DIR",      SPECTRA_DIR),
    ("OBJECT_SPLIT_CSV", OBJECT_SPLIT_CSV),
    ("GROUND_TRUTH_CSV", GROUND_TRUTH_CSV),
    ("BENCHMARK_ROOT",   BENCHMARK_ROOT),
    ("RESULTS_DIR",      RESULTS_DIR),
]
all_ok = True
print("\nPath checks")
for name, path in PATHS_TO_CHECK:
    kind   = "DIR " if path.is_dir()  else "FILE"
    status = "OK     " if path.exists() else "MISSING"
    print(f"  [{kind}] {status}  {name}: {path}")
    if not path.exists():
        all_ok = False

def read_spectrum_file(path):
    """Load a semicolon-delimited wavenumber;reflectance file, return sorted DataFrame."""
    df = pd.read_csv(path, sep=";", header=None,
                     names=["wavenumber", "reflectance"], engine="python")
    if df.shape[1] != 2:
        raise ValueError(f"Expected 2 columns, got {df.shape[1]}: {path}")
    df["wavenumber"]  = pd.to_numeric(df["wavenumber"],  errors="coerce")
    df["reflectance"] = pd.to_numeric(df["reflectance"], errors="coerce")
    if df.isna().any().any():
        raise ValueError(f"Non-numeric values in {path}")
    if df.empty:
        raise ValueError(f"Empty file: {path}")
    return df.sort_values("wavenumber").reset_index(drop=True)

sample_files = [p for p in SPECTRA_DIR.iterdir() if p.is_file()][:3] if SPECTRA_DIR.exists() else []
print(f"\nSpectrum smoke test (first {len(sample_files)} files):")
for p in sample_files:
    try:
        df = read_spectrum_file(p)
        print(f"  OK  {p.name}  ({len(df)} points, "
              f"wn=[{df['wavenumber'].min():.1f}, {df['wavenumber'].max():.1f}])")
    except Exception as exc:
        print(f"  FAIL {p.name}: {exc}")
        all_ok = False

if not all_ok:
    raise RuntimeError("Sanity check FAILED — fix paths before continuing.")
print("\nSanity check PASS")

In [ ]:
# === Cell 6: File Discovery and Parsing ===

# Files are extensionless; filter by FILENAME_REGEX on the stem.
# object_stem (e.g. "HDPE_obj_01") is the join key to the ground-truth CSV;
# group_id has the same value and is the object-level unit for splitting.

def parse_filename(path):
    """Extract material, object_id, region, object_stem, group_id from a filename."""
    match = FILENAME_REGEX.match(Path(path).stem)
    if not match:
        return None
    d   = match.groupdict()
    mat = d["material"].upper()
    oid = d["object_id"].zfill(OBJECT_ID_WIDTH)
    stem = f"{mat}_obj_{oid}"
    return {
        "path":            str(path),
        "filename":        Path(path).name,
        "prefix_material": mat,           # object naming prefix (NOT the label)
        "object_id":       oid,
        "region":          d["region"].lower(),
        "object_stem":     stem,          # join key to ground truth
        "group_id":        stem,          # object-level grouping unit
    }

all_files     = sorted([p for p in SPECTRA_DIR.iterdir() if p.is_file()])
parsed_rows   = []
skipped_files = []
for p in all_files:
    res = parse_filename(p)
    (parsed_rows if res is not None else skipped_files).append(res if res is not None else p.name)

if skipped_files:
    print(f"WARNING: {len(skipped_files)} files did not match FILENAME_REGEX and are excluded:")
    for name in skipped_files[:10]:
        print(f"  - {name}")
    if len(skipped_files) > 10:
        print(f"  ... and {len(skipped_files) - 10} more")

raw_meta_df = pd.DataFrame(parsed_rows)
if raw_meta_df.empty:
    raise RuntimeError(f"No files matched FILENAME_REGEX in {SPECTRA_DIR}.")

print(f"Files discovered:  {len(all_files)}")
print(f"Files parsed:      {len(raw_meta_df)}")
print(f"Unique prefixes:   {sorted(raw_meta_df['prefix_material'].unique())}")
print(f"Unique objects:    {raw_meta_df['group_id'].nunique()}")
print(f"Unique regions:    {sorted(raw_meta_df['region'].unique())}")

print("\nSpectra per region:")
display(raw_meta_df.groupby("region").size().rename("n_spectra"))

In [ ]:
# === Cell 7: Ground Truth and Outer Split ===

# --- (a) Region-specific ground truth ---
# One row per object with the true class of body / label / cap. An empty class
# means that component was excluded from the balanced dataset.
gt_df = pd.read_csv(GROUND_TRUTH_CSV, sep=";", dtype=str)
gt_df.columns = [c.strip() for c in gt_df.columns]
for c in gt_df.columns:
    gt_df[c] = gt_df[c].str.strip().replace({"": np.nan})

print("Ground-truth columns:", list(gt_df.columns))
print(f"Ground-truth objects: {len(gt_df)}")
for region, col in REGION_TO_GT_COLUMN.items():
    vals = sorted(gt_df[col].dropna().unique())
    print(f"  {region:5s} ({col}): classes={vals}  missing={gt_df[col].isna().sum()}")

gt_indexed = gt_df.set_index("filename")

# --- (b) Assign region-specific true class ---
def lookup_true_class(row):
    """Return the ground-truth polymer class for this spectrum's region, or NaN."""
    stem = row["object_stem"]
    col  = REGION_TO_GT_COLUMN[row["region"]]
    if stem not in gt_indexed.index:
        return np.nan
    return gt_indexed.loc[stem, col]

meta_df = raw_meta_df.copy()
meta_df["true_class"] = meta_df.apply(lookup_true_class, axis=1)

# Consistency checks between spectra and ground truth.
no_gt_row   = meta_df[~meta_df["object_stem"].isin(gt_indexed.index)]
no_class    = meta_df[meta_df["object_stem"].isin(gt_indexed.index) & meta_df["true_class"].isna()]
gt_no_spec  = set(gt_indexed.index) - set(meta_df["object_stem"])

print(f"\nSpectra whose object is absent from ground truth: {len(no_gt_row)}")
for n in no_gt_row["filename"].tolist()[:10]:
    print(f"  - {n}")
print(f"Spectra whose region has empty ground-truth class (excluded): {len(no_class)}")
for n in no_class["filename"].tolist()[:10]:
    print(f"  - {n}  (region={no_class.loc[no_class['filename'] == n, 'region'].iloc[0]})")
print(f"Ground-truth objects with no spectra at all: {len(gt_no_spec)}")
for s in sorted(gt_no_spec)[:10]:
    print(f"  - {s}")

# Drop spectra without a ground-truth class (one row per non-empty GT entry remains).
meta_df = meta_df[meta_df["true_class"].notna()].reset_index(drop=True)

# --- (c) Outer split ---
# Objects in object_split.csv keep their segmentation cv/test assignment; other
# objects go to "test" if listed in NEW_TEST_OBJECTS (Config), else to "cv".
split_df = pd.read_csv(OBJECT_SPLIT_CSV, dtype=str)
split_df.columns = [c.strip() for c in split_df.columns]
split_df["material"]    = split_df["material"].str.upper()
split_df["object_id"]   = split_df["object_id"].str.extract(r"(\d+)")[0].str.zfill(OBJECT_ID_WIDTH)
split_df["object_stem"] = split_df["material"] + "_obj_" + split_df["object_id"]
split_df["split"]       = split_df["split"].str.strip().str.lower()

print("\nobject_split.csv split counts (original segmentation split):")
display(split_df.groupby("split")["object_stem"].count().rename("n_objects"))

# Carry-over lookup and forced-test set for objects not in the segmentation split.
carryover_split = dict(zip(split_df["object_stem"], split_df["split"]))
NEW_TEST_STEMS  = {
    f"{mat.upper()}_obj_{int(oid):0{OBJECT_ID_WIDTH}d}" for mat, oid in NEW_TEST_OBJECTS
}

def assign_outer_split(object_stem):
    """cv/test for an object: carry over from segmentation split, else new-object rule."""
    if object_stem in carryover_split:
        return carryover_split[object_stem], "carryover"
    if object_stem in NEW_TEST_STEMS:
        return "test", "new_test"
    return "cv", "new_cv"

_split_origin = meta_df["object_stem"].map(lambda s: assign_outer_split(s))
meta_df["split"]        = _split_origin.map(lambda t: t[0])
meta_df["split_origin"] = _split_origin.map(lambda t: t[1])

# Object-level provenance of the split assignment.
obj_origin = (
    meta_df[["object_stem", "split", "split_origin"]]
    .drop_duplicates()
    .groupby(["split_origin", "split"]).size().rename("n_objects").reset_index()
)
print("\nOuter-split provenance (objects):")
display(obj_origin)

# Warn if a NEW_TEST_OBJECTS entry has no active spectra.
_missing_new_test = sorted(NEW_TEST_STEMS - set(meta_df["object_stem"]))
if _missing_new_test:
    print(f"\nWARNING: NEW_TEST_OBJECTS not found among active spectra: {_missing_new_test}")

# Keep only cv/test rows.
n_before = len(meta_df)
meta_df  = meta_df[meta_df["split"].isin(VALID_SPLITS)].reset_index(drop=True)
if n_before != len(meta_df):
    print(f"\nDropped {n_before - len(meta_df)} spectra outside {VALID_SPLITS}.")

# --- (d) Summaries and sample table export ---
ALL_CLASSES = sorted(meta_df["true_class"].unique())
print(f"\nFinal target classes ({len(ALL_CLASSES)}): {ALL_CLASSES}")

print("\nClass distribution per split:")
display(meta_df.groupby(["split", "true_class"]).size().rename("n_spectra").reset_index())

summary_rows = []
for split_val in ["cv", "test"]:
    sub = meta_df[meta_df["split"] == split_val]
    for region in VALID_REGIONS:
        rsub = sub[sub["region"] == region]
        if len(rsub):
            summary_rows.append({"split": split_val, "region": region,
                                 "n_objects": rsub["group_id"].nunique(), "n_spectra": len(rsub)})
    summary_rows.append({"split": split_val, "region": "ALL",
                         "n_objects": sub["group_id"].nunique(), "n_spectra": len(sub)})
split_summary_df = pd.DataFrame(summary_rows)
print("\nSplit summary (objects and spectra):")
display(split_summary_df)

meta_df.to_csv(SAMPLE_TABLE_CSV, index=False)
split_summary_df.to_csv(SPLIT_SUMMARY_CSV, index=False)
print(f"\nSaved sample table ({len(meta_df)} rows) to: {SAMPLE_TABLE_CSV}")
print(f"Saved split summary to: {SPLIT_SUMMARY_CSV}")


In [ ]:
# === Cell 8: Feature Loading ===

# Load each spectrum into a feature vector; label y = region-specific true_class.
# All spectra must share one wavenumber axis (taken from the first file).

def load_feature_vector(path):
    """Return (reflectance_vector, wavenumber_axis) for one spectrum file."""
    df = read_spectrum_file(path)
    return df["reflectance"].to_numpy(dtype=np.float64), df["wavenumber"].to_numpy()

print("Loading spectra ...")
feature_rows, wavenumber_ref, load_errors, n_features = [], None, [], None
for _, row in meta_df.iterrows():
    try:
        vec, wn = load_feature_vector(row["path"])
        if wavenumber_ref is None:
            wavenumber_ref, n_features = wn, len(vec)
            print(f"  Reference axis: {len(wn)} points, wn=[{wn.min():.1f}, {wn.max():.1f}] cm-1")
        if len(vec) != n_features:
            raise ValueError(f"length {len(vec)} != reference {n_features}")
        feature_rows.append(vec)
    except Exception as exc:
        load_errors.append({"filename": row["filename"], "error": str(exc)})

if load_errors:
    print(f"\nWARNING: {len(load_errors)} files failed to load:")
    for e in load_errors:
        print(f"  - {e['filename']}: {e['error']}")
if not feature_rows:
    raise RuntimeError("No spectra could be loaded.")

X_all = np.vstack(feature_rows)
y_all = meta_df["true_class"].to_numpy()
print(f"\nFeature matrix: {X_all.shape}  ({X_all.shape[0]} spectra x {X_all.shape[1]} points)")
print("Class counts:")
for cls in ALL_CLASSES:
    print(f"  {cls:6s}: {(y_all == cls).sum()}")

# Outer split masks
cv_mask, test_mask = (meta_df["split"] == "cv").to_numpy(), (meta_df["split"] == "test").to_numpy()
X_cv,   y_cv   = X_all[cv_mask],   y_all[cv_mask]
X_test, y_test = X_all[test_mask], y_all[test_mask]
meta_cv   = meta_df[cv_mask].reset_index(drop=True)
meta_test = meta_df[test_mask].reset_index(drop=True)

print(f"\nDevelopment (CV) set: {X_cv.shape[0]} spectra, {meta_cv['group_id'].nunique()} objects")
print(f"Held-out test set:    {X_test.shape[0]} spectra, {meta_test['group_id'].nunique()} objects")

# PCA components must not exceed the smallest fold's training-sample count.
max_pca = max([c for c in PCA_N_COMPONENTS_GRID if c is not None], default=0)
approx_min_train = int(X_cv.shape[0] * (NUM_FOLDS - 1) / NUM_FOLDS)
if max_pca >= approx_min_train:
    print(f"\nNOTE: largest PCA n_components ({max_pca}) is close to the approx. "
          f"per-fold training size (~{approx_min_train}). If PCA raises a size error, "
          f"lower PCA_N_COMPONENTS_GRID in Cell 4.")

In [ ]:
# === Cell 9: CV Fold Construction ===

# Stage A assigns whole objects to NUM_FOLDS folds (material-stratified); stage B
# moves individual spectra so each class has an even per-fold count, which can
# split objects across folds (logged to CSV). The test split stays object-level.

random.seed(RANDOM_SEED)

# --- Stage A: material-stratified object-level folds ---
cv_objects_df = meta_cv[["group_id", "prefix_material"]].drop_duplicates().reset_index(drop=True)
materials     = sorted(cv_objects_df["prefix_material"].unique())

init_fold_of_object = {}
for material in materials:
    obj_ids = sorted(cv_objects_df.loc[cv_objects_df["prefix_material"] == material, "group_id"])
    random.shuffle(obj_ids)
    sizes = [len(obj_ids) // NUM_FOLDS] * NUM_FOLDS
    for i in range(len(obj_ids) % NUM_FOLDS):
        sizes[i] += 1
    start = 0
    for fold_idx, sz in enumerate(sizes):
        for g in obj_ids[start:start + sz]:
            init_fold_of_object[g] = fold_idx
        start += sz

meta_cv = meta_cv.copy()
meta_cv["init_fold"] = meta_cv["group_id"].map(init_fold_of_object)
meta_cv["fold"]      = meta_cv["init_fold"]

def _per_class_fold_counts(df):
    return (df.groupby(["true_class", "fold"]).size()
              .unstack(fill_value=0)
              .reindex(columns=range(NUM_FOLDS), fill_value=0))

print("Per-class spectrum counts per fold BEFORE balancing (object-level only):")
display(_per_class_fold_counts(meta_cv))

# --- Stage B: greedy per-class balancing (deterministic) ---
def _fold_targets(total):
    """Even split of `total` over NUM_FOLDS; remainder to the lowest fold indices."""
    base, rem = divmod(total, NUM_FOLDS)
    return [base + (1 if f < rem else 0) for f in range(NUM_FOLDS)]

reassign_rows = []
for cls in sorted(meta_cv["true_class"].unique()):
    cls_index = meta_cv.index[meta_cv["true_class"] == cls].tolist()
    targets   = _fold_targets(len(cls_index))
    # Deterministic candidate order for which spectrum to move.
    ordered   = meta_cv.loc[cls_index].sort_values(["group_id", "region"]).index.tolist()

    while True:
        counts = [int((meta_cv.loc[ordered, "fold"] == f).sum()) for f in range(NUM_FOLDS)]
        over   = [f for f in range(NUM_FOLDS) if counts[f] > targets[f]]
        under  = [f for f in range(NUM_FOLDS) if counts[f] < targets[f]]
        if not over:
            break
        src, dst = over[0], under[0]
        # Move one spectrum of this class from an over-full to an under-full fold.
        move_idx = next(i for i in ordered if meta_cv.loc[i, "fold"] == src)
        meta_cv.loc[move_idx, "fold"] = dst
        reassign_rows.append({
            "object_stem": meta_cv.loc[move_idx, "object_stem"],
            "filename":    meta_cv.loc[move_idx, "filename"],
            "region":      meta_cv.loc[move_idx, "region"],
            "true_class":  cls,
            "from_fold":   src,
            "to_fold":     dst,
        })

print("\nPer-class spectrum counts per fold AFTER balancing:")
display(_per_class_fold_counts(meta_cv))

# --- Report reassignments and split objects ---
reassign_df = pd.DataFrame(
    reassign_rows,
    columns=["object_stem", "filename", "region", "true_class", "from_fold", "to_fold"],
)
reassign_df.to_csv(FOLD_REASSIGN_CSV, index=False)
print(f"\nSpectra reassigned by the balancing pass: {len(reassign_df)}")
if len(reassign_df):
    display(reassign_df)

# Objects whose spectra span more than one final fold.
obj_fold_spread = (
    meta_cv.groupby("group_id")
    .agg(folds=("fold", lambda s: sorted(set(int(x) for x in s))),
         n_spectra=("fold", "size"))
)
split_objects_df = obj_fold_spread[obj_fold_spread["folds"].map(len) > 1].reset_index()
split_objects_df["folds"] = split_objects_df["folds"].map(lambda fs: ",".join(map(str, fs)))
split_objects_df.to_csv(SPLIT_OBJECTS_CSV, index=False)
print(f"\nObjects split across folds by the balancing pass: {len(split_objects_df)} "
      f"of {meta_cv['group_id'].nunique()} CV objects")
if len(split_objects_df):
    display(split_objects_df)
print(f"Reassignment log saved to: {FOLD_REASSIGN_CSV}")
print(f"Split-object report saved to: {SPLIT_OBJECTS_CSV}")

# --- Save fold assignment and per-fold summary ---
fold_assignment_df = (
    meta_cv[["filename", "object_stem", "group_id", "prefix_material",
             "region", "true_class", "init_fold", "fold"]]
    .assign(reassigned=lambda d: d["init_fold"] != d["fold"])
    .sort_values(["fold", "true_class", "object_stem", "region"])
    .reset_index(drop=True)
)
fold_assignment_df.to_csv(FOLD_ASSIGNMENT_CSV, index=False)

fold_summary_rows = []
for fold_idx in range(NUM_FOLDS):
    val = meta_cv[meta_cv["fold"] == fold_idx]
    trn = meta_cv[meta_cv["fold"] != fold_idx]
    fold_summary_rows.append({
        "fold": fold_idx,
        "train_spectra": len(trn), "val_spectra": len(val),
        "val_objects":   val["group_id"].nunique(),
        "val_objects_also_in_train": int(
            val["group_id"].isin(trn["group_id"]).groupby(val["group_id"]).any().sum()
        ),
    })
print("\nPer-fold development split summary (spectrum-level folds):")
display(pd.DataFrame(fold_summary_rows))
print(f"Fold assignment table saved to: {FOLD_ASSIGNMENT_CSV}")

# --- sklearn (train_idx, val_idx) pairs over meta_cv / X_cv rows ---
CV_FOLD_INDICES = []
for fold_idx in range(NUM_FOLDS):
    val_mask = (meta_cv["fold"] == fold_idx).to_numpy()
    trn_mask = ~val_mask
    CV_FOLD_INDICES.append((np.where(trn_mask)[0], np.where(val_mask)[0]))
print("CV fold indices ready for GridSearchCV.")


In [ ]:
# === Cell 10: Hyperparameter Tuning and CV Evaluation ===

# Per classifier: Pipeline([StandardScaler, PCA, clf]) tuned by GridSearchCV on
# CV_FOLD_INDICES (PCA dims included), multi-metric scoring, refit on macro-F1.
# Per-fold metrics of the best candidate are read from cv_results_.

def compute_metrics(y_true, y_pred):
    """Primary and secondary classification metrics for one prediction set."""
    return {
        "macro-F1":          f1_score(y_true, y_pred, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro-precision":   precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro-recall":      recall_score(y_true, y_pred, average="macro", zero_division=0),
        "accuracy":          accuracy_score(y_true, y_pred),
    }

def build_pipeline(estimator):
    """Build mean-centre -> PCA -> classifier pipeline.

    Spectra are already z-score normalised per sample, so features are only
    mean-centred (with_std=False) and not scaled to unit variance.
    """
    return Pipeline([
        ("scaler", StandardScaler(with_std=False)),   # mean-centre only, no variance scaling
        ("pca",    PCA(random_state=RANDOM_SEED)),
        ("clf",    estimator),
    ])

SCORING = {
    "macro-F1": "f1_macro", "balanced_accuracy": "balanced_accuracy",
    "macro-precision": "precision_macro", "macro-recall": "recall_macro",
    "accuracy": "accuracy",
}
METRIC_KEYS = list(SCORING.keys())

# Save parameter grids as JSON.
def _serialise_grid(grid):
    """Convert a param grid (dict or list of dicts) to a JSON-safe structure."""
    def _val(v):
        if v is None or isinstance(v, bool) or isinstance(v, (int, float)):
            return v
        return str(v)
    def _conv(d):
        return {k: [_val(x) for x in vs] for k, vs in d.items()}
    return [_conv(g) for g in grid] if isinstance(grid, list) else _conv(grid)

with PARAM_GRIDS_JSON.open("w", encoding="utf-8") as f:
    json.dump({n: _serialise_grid(g) for n, (_, g) in PARAM_GRIDS.items()},
              f, indent=2, ensure_ascii=False)
print(f"Hyperparameter grids saved to: {PARAM_GRIDS_JSON}\n")

hpo_results, cv_fold_rows, BEST_ESTIMATORS = [], [], {}

for clf_name, (estimator, param_grid) in PARAM_GRIDS.items():
    print("=" * 70)
    print(f"Classifier: {clf_name}")
    gs = GridSearchCV(
        estimator  = build_pipeline(estimator),
        param_grid = param_grid,
        cv         = CV_FOLD_INDICES,
        scoring    = SCORING,
        refit      = "macro-F1",
        n_jobs     = N_JOBS,
        verbose    = 0,
    )
    gs.fit(X_cv, y_cv)
    BEST_ESTIMATORS[clf_name] = gs.best_estimator_

    print(f"  Best params:      {gs.best_params_}")
    print(f"  Best CV macro-F1: {gs.best_score_:.4f}")

    hpo_results.append({
        "classifier":       clf_name,
        "best_cv_macro_f1": gs.best_score_,
        "best_n_components": gs.best_params_.get("pca__n_components", None),
        "best_params":      str(gs.best_params_),
    })

    best_idx = gs.best_index_
    for fold_idx in range(NUM_FOLDS):
        row = {"classifier": clf_name, "fold": fold_idx}
        for metric in METRIC_KEYS:
            row[metric] = gs.cv_results_[f"split{fold_idx}_test_{metric}"][best_idx]
        cv_fold_rows.append(row)
    print()

cv_fold_df = pd.DataFrame(cv_fold_rows)
agg_rows = []
for clf_name in PARAM_GRIDS:
    sub = cv_fold_df[cv_fold_df["classifier"] == clf_name]
    row = {"classifier": clf_name}
    for m in METRIC_KEYS:
        row[f"{m}_mean"], row[f"{m}_std"] = sub[m].mean(), sub[m].std(ddof=1)
    agg_rows.append(row)
cv_summary_df = pd.DataFrame(agg_rows)

cv_display = cv_summary_df[["classifier"]].copy()
for m in METRIC_KEYS:
    cv_display[m] = cv_summary_df.apply(lambda r: f"{r[f'{m}_mean']:.3f} +/- {r[f'{m}_std']:.3f}", axis=1)
print("CV performance summary (mean +/- std over folds):")
display(cv_display)

hpo_df = pd.DataFrame(hpo_results)
print("\nHPO best parameters and best CV macro-F1:")
display(hpo_df[["classifier", "best_cv_macro_f1", "best_n_components", "best_params"]])

hpo_df.to_csv(HPO_RESULTS_CSV, index=False)
cv_fold_df.to_csv(CV_METRICS_CSV, index=False)
print(f"\nHPO results saved to:     {HPO_RESULTS_CSV}")
print(f"CV fold metrics saved to: {CV_METRICS_CSV}")

# --- Per-class CV metrics per fold ---
# cv_results_ holds only aggregate scores, so each tuned pipeline is cloned, refit
# per fold and scored per class. Model selection is unaffected.
print("\nPer-class CV pass (refitting tuned pipelines per fold) ...")
cv_per_class_rows = []
cv_proba_rows = []  # Brier Score & AUC-ROC per classifier per fold
for clf_name, best_pipe in BEST_ESTIMATORS.items():
    for fold_idx, (trn_idx, val_idx) in enumerate(CV_FOLD_INDICES):
        pipe = clone(best_pipe)
        pipe.fit(X_cv[trn_idx], y_cv[trn_idx])
        y_hat = pipe.predict(X_cv[val_idx])
        f1_per_class = f1_score(y_cv[val_idx], y_hat, labels=ALL_CLASSES,
                                average=None, zero_division=0)
        support = pd.Series(y_cv[val_idx]).value_counts()
        for cls, f1v in zip(ALL_CLASSES, f1_per_class):
            cv_per_class_rows.append({
                "classifier": clf_name, "fold": fold_idx, "class": cls,
                "f1": float(f1v), "support": int(support.get(cls, 0)),
            })

        # Probabilistic metrics (Brier Score & AUC-ROC) from the same refit
        proba_row = {"classifier": clf_name, "fold": fold_idx}
        if hasattr(pipe, "predict_proba"):
            y_proba_fold = pipe.predict_proba(X_cv[val_idx])
            # Align probability columns to ALL_CLASSES (pipe.classes_ may
            # be a subset if a class is missing from the training fold)
            proba_aligned = np.zeros((len(val_idx), len(ALL_CLASSES)))
            for j, cls in enumerate(pipe.classes_):
                col = ALL_CLASSES.index(cls)
                proba_aligned[:, col] = y_proba_fold[:, j]

            # Brier Score (multiclass): mean squared error between one-hot
            # truth and predicted probabilities.  Lower is better.
            y_ohe = label_binarize(y_cv[val_idx], classes=ALL_CLASSES)
            brier = float(np.mean(np.sum((y_ohe - proba_aligned) ** 2, axis=1)))
            proba_row["brier_score"] = brier

            # AUC-ROC (macro OVR): requires >= 2 classes in truth
            n_present = len(set(y_cv[val_idx]))
            if n_present >= 2:
                try:
                    auc = float(roc_auc_score(
                        y_ohe, proba_aligned, multi_class="ovr", average="macro",
                    ))
                except ValueError:
                    auc = np.nan
            else:
                auc = np.nan
            proba_row["AUC-ROC"] = auc
        else:
            proba_row["brier_score"] = np.nan
            proba_row["AUC-ROC"]    = np.nan
        cv_proba_rows.append(proba_row)

# Merge probabilistic metrics into cv_fold_df (used by the result plots and
# the statistical comparison cells).
cv_proba_df = pd.DataFrame(cv_proba_rows)
cv_fold_df = cv_fold_df.merge(cv_proba_df, on=["classifier", "fold"], how="left")

# Add probabilistic metrics to the aggregate summary.
for m in ["brier_score", "AUC-ROC"]:
    for row in agg_rows:
        sub = cv_fold_df[cv_fold_df["classifier"] == row["classifier"]]
        row[f"{m}_mean"] = sub[m].mean()
        row[f"{m}_std"]  = sub[m].std(ddof=1)
cv_summary_df = pd.DataFrame(agg_rows)

# Display the summary including probabilistic metrics.
METRIC_KEYS_EXT = METRIC_KEYS + ["brier_score", "AUC-ROC"]
cv_display_ext = cv_summary_df[["classifier"]].copy()
for m in METRIC_KEYS_EXT:
    cv_display_ext[m] = cv_summary_df.apply(
        lambda r: f"{r[f'{m}_mean']:.4f} +/- {r[f'{m}_std']:.4f}", axis=1,
    )
print("\nCV performance summary including probabilistic metrics:")
display(cv_display_ext)

# Save cv_fold_df again, including brier_score and AUC-ROC.
cv_fold_df.to_csv(CV_METRICS_CSV, index=False)
print(f"Updated CV fold metrics (with Brier Score & AUC-ROC) saved to: {CV_METRICS_CSV}")

cv_per_class_df = pd.DataFrame(cv_per_class_rows)

# Per-class F1 summary across folds (mean +/- std).
cv_per_class_summary = (
    cv_per_class_df.groupby(["classifier", "class"])
    .agg(f1_mean=("f1", "mean"), f1_std=("f1", lambda s: s.std(ddof=1)),
         support_mean=("support", "mean"))
    .reset_index()
)
print("\nPer-class CV F1 (mean +/- std over folds):")
pc_display = cv_per_class_summary.copy()
pc_display["F1"] = pc_display.apply(lambda r: f"{r['f1_mean']:.3f} +/- {r['f1_std']:.3f}", axis=1)
display(pc_display.pivot(index="classifier", columns="class", values="F1"))

cv_per_class_df.to_csv(CV_PER_CLASS_CSV, index=False)
print(f"\nPer-class CV metrics saved to: {CV_PER_CLASS_CSV}")


In [ ]:
# === Cell 11: Final Test Evaluation ===

# BEST_ESTIMATORS are refit on the full CV pool by GridSearchCV;
# each is applied once to the held-out test set.

test_metric_rows, test_pred_rows = [], []
print("Test evaluation (single pass per classifier)\n" + "-" * 70)

for clf_name, best_pipe in BEST_ESTIMATORS.items():
    y_pred  = best_pipe.predict(X_test)
    y_proba = best_pipe.predict_proba(X_test) if hasattr(best_pipe, "predict_proba") else None

    metrics = compute_metrics(y_test, y_pred)

    # Probabilistic metrics on the held-out test set
    if y_proba is not None:
        # Align probability columns to ALL_CLASSES.
        proba_aligned = np.zeros((len(y_test), len(ALL_CLASSES)))
        for j, cls in enumerate(best_pipe.classes_):
            col = ALL_CLASSES.index(cls)
            proba_aligned[:, col] = y_proba[:, j]
        y_ohe_test = label_binarize(y_test, classes=ALL_CLASSES)
        metrics["brier_score"] = float(np.mean(np.sum((y_ohe_test - proba_aligned) ** 2, axis=1)))
        try:
            metrics["AUC-ROC"] = float(roc_auc_score(
                y_ohe_test, proba_aligned, multi_class="ovr", average="macro",
            ))
        except ValueError:
            metrics["AUC-ROC"] = np.nan
    else:
        metrics["brier_score"] = np.nan
        metrics["AUC-ROC"]    = np.nan

    test_metric_rows.append({"classifier": clf_name, **metrics})
    print(f"{clf_name}")
    for k, v in metrics.items():
        print(f"  {k:<22} {v:.4f}")
    print()

    for i, (_, row) in enumerate(meta_test.iterrows()):
        pred_row = {
            "classifier": clf_name, "filename": row["filename"], "group_id": row["group_id"],
            "prefix_material": row["prefix_material"], "region": row["region"],
            "y_true": y_test[i], "y_pred": y_pred[i], "correct": y_test[i] == y_pred[i],
        }
        if y_proba is not None:
            for j, cls in enumerate(best_pipe.classes_):
                pred_row[f"prob_{cls}"] = round(float(y_proba[i, j]), 6)
        test_pred_rows.append(pred_row)

test_metrics_df = pd.DataFrame(test_metric_rows)
print("=" * 70)
print("Test set comparison (held-out objects, single evaluation):")
display(test_metrics_df.round(4))

test_preds_df = pd.DataFrame(test_pred_rows)
print("\nRegion-wise performance per classifier:")
region_df = (
    test_preds_df.groupby(["classifier", "region"])
    .apply(lambda g: pd.Series({
        "n_spectra": len(g),
        "macro-F1":  f1_score(g["y_true"], g["y_pred"], average="macro", zero_division=0),
        "accuracy":  accuracy_score(g["y_true"], g["y_pred"]),
    }), include_groups=False)
    .reset_index()
)
display(region_df.round(4))

test_metrics_df.to_csv(TEST_METRICS_CSV, index=False)
test_preds_df.to_csv(TEST_PREDICTIONS_CSV, index=False)
print(f"\nTest metrics saved to:     {TEST_METRICS_CSV}")
print(f"Test predictions saved to: {TEST_PREDICTIONS_CSV}")

for clf_name, pipe in BEST_ESTIMATORS.items():
    mp = MODELS_DIR / f"{PROJECT_NAME}_{clf_name}_best_estimator.joblib"
    joblib.dump(pipe, mp)
    print(f"Saved {clf_name} estimator to: {mp}")

In [ ]:
# === Cell 12: Result Plots and Decision Boundaries ===
# Reloads metrics, predictions and models from RESULTS_DIR (needs only the Config
# cell). Section 4 also needs the in-memory feature data and is skipped without it.

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from matplotlib.colors import ListedColormap, BoundaryNorm
import joblib

# --- Style config (10 pt at 16 cm LaTeX text width) ---
_TEXT_WIDTH_CM  = 16
_FW             = _TEXT_WIDTH_CM / 2.54          # full width in inches
_FONT_SIZE      = 10
_BAND_COLOR     = "#f0f0f0"

plt.rcParams["font.family"]     = "DejaVu Sans"
plt.rcParams["font.size"]       = _FONT_SIZE
plt.rcParams["axes.titlesize"]  = _FONT_SIZE + 1
plt.rcParams["axes.labelsize"]  = _FONT_SIZE
plt.rcParams["legend.fontsize"] = _FONT_SIZE - 1
plt.rcParams["savefig.dpi"]     = 150

def _add_bands(ax, n):
    """Alternating light-grey / white vertical bands behind bar positions."""
    for i in range(n):
        if i % 2 == 0:
            ax.axvspan(i - 0.5, i + 0.5, color=_BAND_COLOR, zorder=0)


# --- 0. Reload all data from disk ---
# Output PNG paths (defaults if not defined by the Config cell).
CV_COMPARISON_PNG    = globals().get("CV_COMPARISON_PNG",
    RESULTS_DIR / f"{PROJECT_NAME}_cv_comparison.png")
TEST_COMPARISON_PNG  = globals().get("TEST_COMPARISON_PNG",
    RESULTS_DIR / f"{PROJECT_NAME}_test_comparison.png")
BRIER_SCORE_PNG      = globals().get("BRIER_SCORE_PNG",
    RESULTS_DIR / f"{PROJECT_NAME}_brier_score.png")
AUC_ROC_PNG          = globals().get("AUC_ROC_PNG",
    RESULTS_DIR / f"{PROJECT_NAME}_auc_roc.png")
CONFUSION_MATRIX_PNG = globals().get("CONFUSION_MATRIX_PNG",
    RESULTS_DIR / f"{PROJECT_NAME}_confusion_matrices.png")
DECISION_BOUNDARY_PNG = RESULTS_DIR / f"{PROJECT_NAME}_decision_boundaries.png"
COMBINED_F1_BRIER_PNG = RESULTS_DIR / f"{PROJECT_NAME}_test_f1_brier_combined.png"

# --- Result CSV paths ---
_cv_csv  = RESULTS_DIR / f"{PROJECT_NAME}_cv_metrics.csv"
_te_csv  = RESULTS_DIR / f"{PROJECT_NAME}_test_metrics.csv"
_pred_csv = RESULTS_DIR / f"{PROJECT_NAME}_test_predictions.csv"

if not _cv_csv.exists():
    raise FileNotFoundError(f"CV metrics not found: {_cv_csv}\n"
                            "Run Cells 5-10 first to produce the benchmark artefacts.")
if not _te_csv.exists():
    raise FileNotFoundError(f"Test metrics not found: {_te_csv}\n"
                            "Run Cells 5-11 first to produce the benchmark artefacts.")

cv_fold_df     = pd.read_csv(_cv_csv)
test_metrics_df = pd.read_csv(_te_csv)

# Build cv_summary_df: per-classifier mean +/- std of key metrics over folds
_cv_agg_cols = [c for c in cv_fold_df.columns
                if c not in ("classifier", "fold")]
cv_summary_df = (cv_fold_df.groupby("classifier")[_cv_agg_cols]
                 .agg(["mean", lambda s: s.std(ddof=1)])
                 .reset_index())
# Flatten MultiIndex columns: metric_mean, metric_std
cv_summary_df.columns = [
    col[0] if col[1] == "" else f"{col[0]}_{col[1]}" if col[1] == "mean"
    else f"{col[0]}_std"
    for col in cv_summary_df.columns
]
# Rename lambda aggregation columns to "std".
cv_summary_df.columns = [c.replace("<lambda_0>", "std").replace("<lambda>", "std")
                         for c in cv_summary_df.columns]

# cv_agg frame plotted in section 1
cv_agg = cv_summary_df[["classifier", "macro-F1_mean", "macro-F1_std"]].copy()
cv_agg.columns = ["classifier", "mean", "std"]

# Load predictions for confusion matrices
test_preds_df = pd.read_csv(_pred_csv) if _pred_csv.exists() else pd.DataFrame()

# Derive ALL_CLASSES from predictions or test metrics
if not test_preds_df.empty:
    ALL_CLASSES = sorted(test_preds_df["y_true"].unique())
else:
    ALL_CLASSES = sorted(test_metrics_df["classifier"].unique())  # fallback

# Classifier name list (for confusion matrices & decision boundaries)
clf_names = test_metrics_df["classifier"].tolist()

# Load fitted estimators from disk (for decision boundaries)
_models_dir = RESULTS_DIR / "models"
_BEST_ESTIMATORS_DISK = {}
if _models_dir.is_dir():
    for clf in clf_names:
        _jl = _models_dir / f"{PROJECT_NAME}_{clf}_best_estimator.joblib"
        if _jl.exists():
            try:
                _BEST_ESTIMATORS_DISK[clf] = joblib.load(_jl)
            except Exception as e:
                print(f"  [warn] Could not load {_jl.name}: {e}")

print(f"Loaded CV metrics:  {len(cv_fold_df)} rows from {_cv_csv.name}")
print(f"Loaded test metrics: {len(test_metrics_df)} rows from {_te_csv.name}")
if not test_preds_df.empty:
    print(f"Loaded predictions: {len(test_preds_df)} rows from {_pred_csv.name}")
print(f"Loaded estimators:  {list(_BEST_ESTIMATORS_DISK.keys()) or 'none'}")


# --- 1. CV Macro-F1 Comparison (Mean +/- Std) ---
fig_cv, ax = plt.subplots(figsize=(_FW, 3.65))
x = np.arange(len(cv_agg))
_add_bands(ax, len(x))
ax.bar(x, cv_agg["mean"], yerr=cv_agg["std"], color="tab:blue", alpha=0.85, capsize=5, width=0.5)
ax.set_xticks(x); ax.set_xticklabels(cv_agg["classifier"]); ax.set_ylim(0, 1.05)
ax.set_ylabel("macro-F1"); ax.grid(axis="y", alpha=0.3)
ax.set_title("CV Development Performance — Macro-F1")
for xi, mv in zip(x, cv_agg["mean"]):
    ax.text(xi, mv + 0.01, f"{mv:.3f}", ha="center", va="bottom", fontsize=_FONT_SIZE - 1)
plt.tight_layout(); fig_cv.savefig(CV_COMPARISON_PNG, bbox_inches="tight", dpi=150); plt.show()
print("Saved:", CV_COMPARISON_PNG)

# --- 2. Test Macro-F1 Comparison ---
fig_test, ax = plt.subplots(figsize=(_FW, 3.65))
x = np.arange(len(test_metrics_df))
_add_bands(ax, len(x))
ax.bar(x, test_metrics_df["macro-F1"], color="tab:orange", alpha=0.85, width=0.5)
ax.set_xticks(x); ax.set_xticklabels(test_metrics_df["classifier"]); ax.set_ylim(0, 1.05)
ax.set_ylabel("macro-F1"); ax.grid(axis="y", alpha=0.3)
ax.set_title("Test Set Performance — Macro-F1")
for xi, v in zip(x, test_metrics_df["macro-F1"]):
    ax.text(xi, v + 0.01, f"{v:.3f}", ha="center", va="bottom", fontsize=_FONT_SIZE - 1)
plt.tight_layout(); fig_test.savefig(TEST_COMPARISON_PNG, bbox_inches="tight", dpi=150); plt.show()
print("Saved:", TEST_COMPARISON_PNG)

# --- 2b. Brier Score Comparison (CV Mean +/- Std, Plus Test) ---
fig_bs, axes_bs = plt.subplots(1, 2, figsize=(_FW, 3.65), sharey=True)

bs_cv_means = cv_summary_df["brier_score_mean"].values
bs_cv_stds  = cv_summary_df["brier_score_std"].values
x_bs = np.arange(len(cv_summary_df))
_add_bands(axes_bs[0], len(x_bs))
axes_bs[0].bar(
    x_bs, bs_cv_means, yerr=bs_cv_stds,
    color="tab:green", alpha=0.85, capsize=5, width=0.5,
)
axes_bs[0].set_xticks(x_bs)
axes_bs[0].set_xticklabels(cv_summary_df["classifier"])
axes_bs[0].set_ylabel("Brier Score")
axes_bs[0].set_title("CV Brier Score")
axes_bs[0].grid(axis="y", alpha=0.3)
for xi, mv in zip(x_bs, bs_cv_means):
    axes_bs[0].text(xi, mv + 0.001, f"{mv:.4f}", ha="center", va="bottom", fontsize=_FONT_SIZE - 1)

bs_test = test_metrics_df["brier_score"].values
_add_bands(axes_bs[1], len(x_bs))
axes_bs[1].bar(
    x_bs, bs_test,
    color="tab:red", alpha=0.85, width=0.5,
)
axes_bs[1].set_xticks(x_bs)
axes_bs[1].set_xticklabels(test_metrics_df["classifier"])
axes_bs[1].set_title("Test Brier Score")
axes_bs[1].grid(axis="y", alpha=0.3)
for xi, v in zip(x_bs, bs_test):
    axes_bs[1].text(xi, v + 0.001, f"{v:.4f}", ha="center", va="bottom", fontsize=_FONT_SIZE - 1)

fig_bs.suptitle("Brier Score — Probability Calibration Quality", fontsize=_FONT_SIZE + 1)
plt.tight_layout()
fig_bs.savefig(BRIER_SCORE_PNG, bbox_inches="tight", dpi=150)
plt.show()
print("Saved:", BRIER_SCORE_PNG)

# --- 2c. AUC-ROC Comparison (CV Mean +/- Std, Plus Test) ---
fig_auc, axes_auc = plt.subplots(1, 2, figsize=(_FW, 3.65), sharey=True)

auc_cv_means = cv_summary_df["AUC-ROC_mean"].values
auc_cv_stds  = cv_summary_df["AUC-ROC_std"].values
_add_bands(axes_auc[0], len(x_bs))
axes_auc[0].bar(
    x_bs, auc_cv_means, yerr=auc_cv_stds,
    color="tab:purple", alpha=0.85, capsize=5, width=0.5,
)
axes_auc[0].set_xticks(x_bs)
axes_auc[0].set_xticklabels(cv_summary_df["classifier"])
axes_auc[0].set_ylim(0, 1.05)
axes_auc[0].set_ylabel("AUC-ROC (macro OVR)")
axes_auc[0].set_title("CV AUC-ROC")
axes_auc[0].grid(axis="y", alpha=0.3)
for xi, mv in zip(x_bs, auc_cv_means):
    axes_auc[0].text(xi, mv + 0.005, f"{mv:.4f}", ha="center", va="bottom", fontsize=_FONT_SIZE - 1)

auc_test = test_metrics_df["AUC-ROC"].values
_add_bands(axes_auc[1], len(x_bs))
axes_auc[1].bar(
    x_bs, auc_test,
    color="tab:cyan", alpha=0.85, width=0.5,
)
axes_auc[1].set_xticks(x_bs)
axes_auc[1].set_xticklabels(test_metrics_df["classifier"])
axes_auc[1].set_ylim(0, 1.05)
axes_auc[1].set_title("Test AUC-ROC")
axes_auc[1].grid(axis="y", alpha=0.3)
for xi, v in zip(x_bs, auc_test):
    axes_auc[1].text(xi, v + 0.005, f"{v:.4f}", ha="center", va="bottom", fontsize=_FONT_SIZE - 1)

fig_auc.suptitle("AUC-ROC — Threshold-Independent Class Discrimination", fontsize=_FONT_SIZE + 1)
plt.tight_layout()
fig_auc.savefig(AUC_ROC_PNG, bbox_inches="tight", dpi=150)
plt.show()
print("Saved:", AUC_ROC_PNG)

# --- 2d. Combined Test Macro-F1 & Brier Score (thesis main figure) ---
_clf_names_sorted = test_metrics_df["classifier"].tolist()
_n_clfs = len(_clf_names_sorted)
_clf_colors = [plt.cm.tab10(i) for i in range(_n_clfs)]

_panel_w = _FW / 2
fig_comb, (ax_f1, ax_bs) = plt.subplots(1, 2, figsize=(_FW, _panel_w))  # square panels
x_comb = np.arange(_n_clfs)
bar_w  = 0.55

# Left: Test Macro-F1
_add_bands(ax_f1, _n_clfs)
_f1_vals = [test_metrics_df.loc[test_metrics_df["classifier"] == clf, "macro-F1"].iloc[0]
            for clf in _clf_names_sorted]
for i, (clf, col, val) in enumerate(zip(_clf_names_sorted, _clf_colors, _f1_vals)):
    ax_f1.bar(i, val, width=bar_w, color=col, alpha=0.85, edgecolor="black", linewidth=0.4)
    ax_f1.text(i, val + 0.008, f"{val:.3f}", ha="center", va="bottom",
               fontsize=_FONT_SIZE - 1, fontweight="bold")
ax_f1.set_xticks(x_comb)
ax_f1.set_xticklabels(_clf_names_sorted)
ax_f1.set_ylim(0, max(_f1_vals) * 1.18)  # headroom so bold labels don't border the top edge
ax_f1.set_ylabel("Macro-F1")
ax_f1.set_title("Test Macro-F1")
ax_f1.set_box_aspect(1)  # square panel
ax_f1.grid(axis="y", alpha=0.3)

# Right: Test Brier Score
_add_bands(ax_bs, _n_clfs)
_bs_vals = [test_metrics_df.loc[test_metrics_df["classifier"] == clf, "brier_score"].iloc[0]
            for clf in _clf_names_sorted]
for i, (clf, col, val) in enumerate(zip(_clf_names_sorted, _clf_colors, _bs_vals)):
    ax_bs.bar(i, val, width=bar_w, color=col, alpha=0.85, edgecolor="black", linewidth=0.4)
    ax_bs.text(i, val + 0.001, f"{val:.4f}", ha="center", va="bottom",
               fontsize=_FONT_SIZE - 1, fontweight="bold")
ax_bs.set_xticks(x_comb)
ax_bs.set_xticklabels(_clf_names_sorted)
ax_bs.set_ylim(0, max(_bs_vals) * 1.18)  # headroom so bold labels don't border the top edge
ax_bs.set_ylabel("Brier Score")
ax_bs.set_title("Test Brier Score")
ax_bs.set_box_aspect(1)  # square panel
ax_bs.grid(axis="y", alpha=0.3)

fig_comb.suptitle("Test Set Performance — Macro-F1 and Brier Score", fontsize=_FONT_SIZE + 1)
plt.tight_layout()
fig_comb.savefig(COMBINED_F1_BRIER_PNG, bbox_inches="tight", dpi=150)
plt.show()
print("Saved:", COMBINED_F1_BRIER_PNG)

# --- 3. Confusion Matrices (Row-Normalised), One Panel per Classifier ---
if not test_preds_df.empty:
    n_cols    = 2
    n_rows    = (len(clf_names) + n_cols - 1) // n_cols   # ceil(5/2) = 3
    _panel_w  = _FW / n_cols
    fig_cm, axes = plt.subplots(n_rows, n_cols, figsize=(_FW, _panel_w * n_rows))
    axes = np.array(axes).reshape(-1)
    for ax, clf_name in zip(axes, clf_names):
        preds = test_preds_df[test_preds_df["classifier"] == clf_name]
        cm = confusion_matrix(preds["y_true"], preds["y_pred"],
                              labels=ALL_CLASSES, normalize="true")
        sns.heatmap(cm, annot=True, fmt=".2f", ax=ax,
                    xticklabels=ALL_CLASSES, yticklabels=ALL_CLASSES,
                    cmap="Blues", linewidths=0.5, vmin=0, vmax=1, cbar=False,
                    annot_kws={"fontsize": _FONT_SIZE - 2}, square=True)
        f1v = test_metrics_df.loc[
            test_metrics_df["classifier"] == clf_name, "macro-F1"
        ].iloc[0]
        ax.set_title(f"{clf_name} (macro-F1 = {f1v:.3f})", fontsize=_FONT_SIZE)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.tick_params(axis="x", rotation=45, labelsize=_FONT_SIZE - 2)
        ax.tick_params(axis="y", labelsize=_FONT_SIZE - 2)
        ax.set_box_aspect(1)
    # Hide unused panels.
    for ax in axes[len(clf_names):]:
        ax.axis("off")
    fig_cm.suptitle("Confusion Matrices — Held-Out Test Set (Row-Normalised)",
                    fontsize=_FONT_SIZE + 1, y=1.01)
    plt.tight_layout()
    fig_cm.savefig(CONFUSION_MATRIX_PNG, bbox_inches="tight", dpi=150)
    plt.show()
    print("Saved:", CONFUSION_MATRIX_PNG)
else:
    print("No test predictions found — skipping confusion matrices.")

# --- 4. Decision Boundary Plots (2D PCA Projection) ---
# Requires in-memory X_cv, X_test, y_cv, y_test (Feature Loading cell) and fitted
# estimators; skipped otherwise.

# In-memory estimators first, else those loaded from disk.
_estimators = globals().get("BEST_ESTIMATORS", _BEST_ESTIMATORS_DISK)

try:
    _X_cv    = X_cv       # noqa: F821 — defined by the Feature Loading cell
    _X_test  = X_test     # noqa: F821
    _y_cv    = y_cv       # noqa: F821
    _y_test  = y_test     # noqa: F821
    _HAS_FEATURE_DATA = True
except NameError:
    _HAS_FEATURE_DATA = False

if not _HAS_FEATURE_DATA or not _estimators:
    print("\nDecision boundary plots SKIPPED — feature data (Cells 6-9) "
          "and/or fitted estimators (Cell 10) not available.")
    print("  Re-run Cells 5-11, then re-run this cell to include decision boundaries.")
else:
    RANDOM_SEED = globals().get("RANDOM_SEED", 42)

    # --- 4a. Project to 2D PCA ---
    scaler_2d = StandardScaler(with_std=False).fit(_X_cv)
    pca_2d    = PCA(n_components=2, random_state=RANDOM_SEED)
    X_cv_2d   = pca_2d.fit_transform(scaler_2d.transform(_X_cv))
    X_test_2d = pca_2d.transform(scaler_2d.transform(_X_test))

    ev = pca_2d.explained_variance_ratio_
    print(f"\nDecision boundary visualisation — 2D PCA projection")
    print(f"  Explained variance: PC1={ev[0]:.3f}, PC2={ev[1]:.3f}, total={ev.sum():.3f}")

    # --- 4b. Colour setup ---
    n_classes    = len(ALL_CLASSES)
    class_colors = [plt.cm.tab10(i) for i in range(n_classes)]
    cm_bg        = ListedColormap(class_colors)
    cm_bright    = ListedColormap(class_colors)

    label_to_int = {c: i for i, c in enumerate(ALL_CLASSES)}
    y_cv_int     = np.array([label_to_int[c] for c in _y_cv])
    y_test_int   = np.array([label_to_int[c] for c in _y_test])

    # --- 4c. One subplot per classifier ---
    n_cols_db = 3
    n_rows_db = (len(clf_names) + n_cols_db - 1) // n_cols_db
    fig_db, axes_db = plt.subplots(n_rows_db, n_cols_db,
                                   figsize=(_FW, (_FW / n_cols_db * 0.9) * n_rows_db))
    axes_db = np.array(axes_db).reshape(-1)

    x_min = min(X_cv_2d[:, 0].min(), X_test_2d[:, 0].min()) - 0.5
    x_max = max(X_cv_2d[:, 0].max(), X_test_2d[:, 0].max()) + 0.5
    y_min = min(X_cv_2d[:, 1].min(), X_test_2d[:, 1].min()) - 0.5
    y_max = max(X_cv_2d[:, 1].max(), X_test_2d[:, 1].max()) + 0.5

    for ax, clf_name in zip(axes_db, clf_names):
        best_clf_step = _estimators[clf_name].named_steps["clf"]
        clf_2d = clone(best_clf_step)
        clf_2d.fit(X_cv_2d, _y_cv)

        _res  = 200
        _xx, _yy = np.meshgrid(np.linspace(x_min, x_max, _res),
                                np.linspace(y_min, y_max, _res))
        _Z = np.array([label_to_int[c]
                        for c in clf_2d.predict(np.c_[_xx.ravel(), _yy.ravel()])])
        _Z = _Z.reshape(_xx.shape)
        _norm = BoundaryNorm(np.arange(-0.5, n_classes, 1.0), n_classes)
        ax.pcolormesh(_xx, _yy, _Z, cmap=cm_bg, norm=_norm, alpha=0.35, zorder=1)

        ax.scatter(X_cv_2d[:, 0], X_cv_2d[:, 1],
                   c=y_cv_int, cmap=cm_bright,
                   edgecolors="k", s=30, linewidths=0.5,
                   vmin=0, vmax=n_classes - 1, zorder=3)
        ax.scatter(X_test_2d[:, 0], X_test_2d[:, 1],
                   c=y_test_int, cmap=cm_bright,
                   edgecolors="k", s=55, linewidths=0.5,
                   alpha=0.6, marker="D",
                   vmin=0, vmax=n_classes - 1, zorder=4)

        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)

        score_2d = f1_score(_y_test, clf_2d.predict(X_test_2d),
                            average="macro", zero_division=0)
        f1_full  = test_metrics_df.loc[
            test_metrics_df["classifier"] == clf_name, "macro-F1"
        ].iloc[0]
        ax.set_title(f"{clf_name}\n2D F1 = {score_2d:.3f} | Full F1 = {f1_full:.3f}",
                     fontsize=_FONT_SIZE)
        ax.set_xlabel(f"PC 1 ({ev[0]:.1%} var)")
        ax.set_ylabel(f"PC 2 ({ev[1]:.1%} var)")

    for ax in axes_db[len(clf_names):]:
        ax.axis("off")

    legend_handles = [
        plt.Line2D([0], [0], marker="o", color="w",
                   markerfacecolor=class_colors[i], markersize=8,
                   markeredgecolor="k", markeredgewidth=0.5,
                   label=ALL_CLASSES[i])
        for i in range(n_classes)
    ]
    legend_handles.append(
        plt.Line2D([0], [0], marker="o", color="w",
                   markerfacecolor="grey", markersize=8,
                   markeredgecolor="k", markeredgewidth=0.5,
                   label="Train (CV Pool)")
    )
    legend_handles.append(
        plt.Line2D([0], [0], marker="D", color="w",
                   markerfacecolor="grey", markersize=8,
                   markeredgecolor="k", markeredgewidth=0.5,
                   alpha=0.6, label="Test (Held-Out)")
    )
    fig_db.legend(handles=legend_handles, loc="lower center",
                  ncol=min(n_classes + 2, 8), fontsize=_FONT_SIZE - 1,
                  bbox_to_anchor=(0.5, -0.04))

    fig_db.suptitle(
        f"Decision Boundaries — 2D PCA Projection "
        f"(Explained Variance: {ev.sum():.1%})",
        fontsize=_FONT_SIZE + 1, y=1.01,
    )
    plt.tight_layout()
    fig_db.savefig(DECISION_BOUNDARY_PNG, bbox_inches="tight", dpi=150)
    plt.show()
    print("Saved:", DECISION_BOUNDARY_PNG)

# --- 5. Final Ranking Printout ---
print("\n" + "=" * 70 + "\nBENCHMARK COMPLETE\n" + "=" * 70)
print("\nCV macro-F1 (mean +/- std):")
for _, r in cv_summary_df.iterrows():
    print(f"  {r['classifier']:<5} {r['macro-F1_mean']:.3f} +/- {r['macro-F1_std']:.3f}")
print("\nTest macro-F1 (ranked):")
for _, r in test_metrics_df.sort_values("macro-F1", ascending=False).iterrows():
    print(f"  {r['classifier']:<5} F1={r['macro-F1']:.3f}  BalAcc={r['balanced_accuracy']:.3f}  Acc={r['accuracy']:.3f}  Brier={r['brier_score']:.4f}  AUC={r['AUC-ROC']:.4f}")
best = test_metrics_df.sort_values("macro-F1", ascending=False).iloc[0]
print(f"\nBest test classifier: {best['classifier']} (macro-F1={best['macro-F1']:.3f})")
print("\nOutputs written to:", RESULTS_DIR)

In [ ]:
# === Cell 13: Statistical Classifier Comparison ===
# Friedman test blocked on CV fold, pairwise Wilcoxon signed-rank (Holm-Bonferroni)
# and bootstrap CIs on cv_fold_df (reloaded from CV_METRICS_CSV if not in memory).
# n = NUM_FOLDS observations per classifier; tests have low power.

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display, clear_output

# --- Resolve the per-fold CV metrics table (in-memory first, else from CSV) ---
def _resolve_cv_fold_df():
    g = globals()
    if isinstance(g.get("cv_fold_df"), pd.DataFrame) and not g["cv_fold_df"].empty:
        return g["cv_fold_df"].copy()
    if "CV_METRICS_CSV" in g:
        csv_path = Path(g["CV_METRICS_CSV"])
    elif "RESULTS_DIR" in g and "PROJECT_NAME" in g:
        csv_path = Path(g["RESULTS_DIR"]) / f"{g['PROJECT_NAME']}_cv_metrics.csv"
    else:
        raise RuntimeError(
            "Could not locate the per-fold CV metrics.  Run Cell 10 (CV "
            "evaluation) first, or define CV_METRICS_CSV (or RESULTS_DIR + "
            "PROJECT_NAME) so the table can be reloaded."
        )
    if not csv_path.exists():
        raise RuntimeError(
            f"cv_fold_df is not in memory and the CSV was not found at:\n  {csv_path}\n"
            "Run Cell 10 (Hyperparameter tuning & CV evaluation) first."
        )
    return pd.read_csv(csv_path)

_cv_fold_df = _resolve_cv_fold_df()

if not {"classifier", "fold"}.issubset(_cv_fold_df.columns):
    raise RuntimeError(
        "Expected columns 'classifier' and 'fold' in the CV metrics table; "
        f"found: {list(_cv_fold_df.columns)}"
    )

# Metric columns = everything that is not an id column; primary metric first
_METRIC_COLS = [c for c in _cv_fold_df.columns if c not in ("classifier", "fold")]
_PRIMARY = globals().get("PRIMARY_METRIC", "macro-F1")
_METRICS = ([_PRIMARY] if _PRIMARY in _METRIC_COLS else []) + \
           [c for c in _METRIC_COLS if c != _PRIMARY]
if not _METRICS:
    raise RuntimeError("No metric columns found in the CV metrics table.")

# Classifier order: first appearance in the table (preserves PARAM_GRIDS order)
_CLASSIFIER_ORDER = list(dict.fromkeys(_cv_fold_df["classifier"].tolist()))

# --- Core helpers ---
def _is_time_metric(metric_name):
    """True for metrics where lower is better (latency, Brier Score)."""
    m = str(metric_name).lower()
    return any(tok in m for tok in ["time", "latency", "ms", "sec", "s/", "speed", "brier"])

def _build_long_df(metric_name):
    """Long-form (classifier, fold, value) for one metric."""
    df = _cv_fold_df[["classifier", "fold", metric_name]].copy()
    df = df.rename(columns={metric_name: "value"})
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df.dropna(subset=["value"])

def _build_block_matrix(df, classifier_order):
    """
    Reshape long df into a (n_folds, n_classifiers) matrix for Friedman / Wilcoxon.
    Returns (matrix, folds, classifiers) or (None, None, None) if the design is
    not a complete block (any cell missing).
    """
    wide = (df.pivot_table(index="fold", columns="classifier",
                           values="value", aggfunc="mean")
              .sort_index())
    cols = [c for c in classifier_order if c in wide.columns]
    wide = wide[cols]
    if wide.isna().any().any() or wide.shape[0] < 2 or wide.shape[1] < 2:
        return None, None, None
    return wide.values, list(wide.index), list(wide.columns)

def _friedman_test(matrix):
    """Friedman chi-square test on a (n_blocks, n_groups) matrix."""
    if matrix is None or matrix.shape[1] < 2:
        return None
    args = [matrix[:, j] for j in range(matrix.shape[1])]
    try:
        stat, p = stats.friedmanchisquare(*args)
    except ValueError as e:
        return {"error": str(e)}
    n_blocks, n_groups = matrix.shape
    # Kendall's W effect size: W = chi2 / (n_blocks * (n_groups - 1))
    W = float(stat) / (n_blocks * (n_groups - 1)) if (n_blocks > 0 and n_groups > 1) else np.nan
    return {"stat": float(stat), "p": float(p), "n_blocks": int(n_blocks),
            "n_groups": int(n_groups), "kendall_w": float(W)}

def _bootstrap_paired_ci(diff_values, alpha=0.05, n_boot=5000, seed=0):
    """Percentile bootstrap CI for the mean of paired differences."""
    if len(diff_values) == 0:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    n = len(diff_values)
    idx = rng.integers(0, n, size=(n_boot, n))
    boot_means = np.asarray(diff_values)[idx].mean(axis=1)
    return (float(np.quantile(boot_means, alpha / 2.0)),
            float(np.quantile(boot_means, 1 - alpha / 2.0)))

def _wilcoxon_p(a, b):
    """
    Two-sided Wilcoxon signed-rank p-value.
    Tries the 'method' argument (newer scipy), then 'mode' (older scipy).
    """
    diff = np.asarray(a) - np.asarray(b)
    if np.all(diff == 0):
        return 1.0
    for kwargs in ({"method": "auto"}, {"mode": "auto"}):
        try:
            return float(stats.wilcoxon(a, b, zero_method="wilcox", correction=False,
                                        alternative="two-sided", **kwargs)[1])
        except TypeError:
            continue
        except ValueError:
            return np.nan
    return np.nan

def _wilcoxon_pairwise(matrix, classifiers, alpha):
    """Pairwise Wilcoxon signed-rank on per-fold differences, Holm-Bonferroni adjusted."""
    rows = []
    for i, c_a in enumerate(classifiers):
        for j in range(i + 1, len(classifiers)):
            c_b = classifiers[j]
            a, b = matrix[:, i], matrix[:, j]
            diff = a - b
            p_raw = _wilcoxon_p(a, b)
            ci_low, ci_high = _bootstrap_paired_ci(diff, alpha=alpha, n_boot=5000)
            rows.append({
                "classifier_A": c_a, "classifier_B": c_b,
                "mean_diff": float(np.mean(diff)),
                "ci_low": ci_low, "ci_high": ci_high,
                "p_value_raw": float(p_raw) if not np.isnan(p_raw) else np.nan,
            })
    if not rows:
        return pd.DataFrame()
    df_p = pd.DataFrame(rows).sort_values("p_value_raw", na_position="last").reset_index(drop=True)
    # Holm-Bonferroni across all (k choose 2) pairs
    m_total = int(df_p["p_value_raw"].notna().sum())
    p_adj, rank_i = [], 0
    for _, row in df_p.iterrows():
        p_raw = row["p_value_raw"]
        if np.isnan(p_raw):
            p_adj.append(np.nan)
            continue
        rank_i += 1
        p_adj.append(min(1.0, p_raw * (m_total - rank_i + 1)))
    df_p["p_value_adj"] = p_adj
    df_p["significant"] = df_p["p_value_adj"].apply(
        lambda v: (False if (v is None or (isinstance(v, float) and np.isnan(v))) else v < alpha)
    )
    return df_p[["classifier_A", "classifier_B", "mean_diff",
                 "ci_low", "ci_high", "p_value_adj", "significant"]]

def _plot_ci_pairs(df_pairs):
    """Horizontal CI plot of pairwise mean differences (95% bootstrap CIs)."""
    if df_pairs is None or df_pairs.empty:
        print("No pairwise comparisons to plot.")
        return
    rows = df_pairs.reset_index(drop=True)
    y_pos = range(len(rows))
    lowers, uppers, means = rows["ci_low"].values, rows["ci_high"].values, rows["mean_diff"].values

    plt.figure(figsize=(8, max(4, len(rows) * 0.5)))
    plt.hlines(y_pos, lowers, uppers, color="tab:blue")
    plt.plot(lowers, y_pos, "o", color="tab:blue")
    plt.plot(uppers, y_pos, "o", color="tab:blue")
    plt.plot(means, y_pos, "x", color="tab:red", markersize=10, markeredgewidth=2)
    plt.vlines(0, -1, len(rows), linestyles="dashed", color="grey")
    labels = [f"{a} - {b}" for a, b in zip(rows["classifier_A"], rows["classifier_B"])]
    plt.gca().set_yticks(list(y_pos))
    plt.gca().set_yticklabels(labels)
    plt.xlabel("Mean difference (95% bootstrap CI)")
    plt.title("Pairwise comparisons (Wilcoxon signed-rank, Holm-Bonferroni; CIs via bootstrap)")
    plt.grid(axis="x", linestyle=":", alpha=0.5)
    plt.tight_layout()
    plt.show()

def _build_text_summary(summary_df, pairwise_df, metric_name, alpha, higher_is_better, friedman_p):
    """Text summary of the best classifier and significant pairwise differences."""
    if summary_df.empty:
        return "No data available for summary."
    summary_sorted = summary_df.sort_values("mean", ascending=not higher_is_better)
    best = summary_sorted.iloc[0]["classifier"]
    p_str = f"{friedman_p:.4g}" if friedman_p is not None and not np.isnan(friedman_p) else "n/a"

    if pairwise_df is None or pairwise_df.empty:
        return (f"For metric {metric_name} (CV folds), classifier {best} has the best "
                f"mean. Friedman p = {p_str}.")

    better_than, not_better = [], []
    for other in summary_df["classifier"].unique():
        if other == best:
            continue
        mask1 = (pairwise_df["classifier_A"] == best) & (pairwise_df["classifier_B"] == other)
        mask2 = (pairwise_df["classifier_A"] == other) & (pairwise_df["classifier_B"] == best)
        if mask1.any():
            row = pairwise_df.loc[mask1].iloc[0]; diff = float(row["mean_diff"])
        elif mask2.any():
            row = pairwise_df.loc[mask2].iloc[0]; diff = -float(row["mean_diff"])
        else:
            not_better.append(other); continue
        if not row["significant"]:
            not_better.append(other); continue
        if higher_is_better:
            (better_than if diff > 0 else not_better).append(other)
        else:
            (better_than if diff < 0 else not_better).append(other)

    if better_than:
        better_list = ", ".join(sorted(better_than))
        if not_better:
            not_list = ", ".join(sorted(not_better))
            return (f"For metric {metric_name} (CV folds), classifier {best} has the best "
                    f"mean and is significantly better than {better_list} "
                    f"(alpha = {alpha:.3f}), but not significantly different from {not_list}. "
                    f"Friedman p = {p_str}.")
        return (f"For metric {metric_name} (CV folds), classifier {best} has the best mean "
                f"and is significantly better than all other classifiers considered "
                f"(alpha = {alpha:.3f}). Friedman p = {p_str}.")
    return (f"For metric {metric_name} (CV folds), classifier {best} has the best mean, but "
            f"no statistically significant pairwise differences were found at "
            f"alpha = {alpha:.3f}. Friedman p = {p_str}.")

def _analyze_one_subset(df, metric_name, alpha):
    """Summary + Friedman + Wilcoxon + bootstrap CIs for the CV-fold data."""
    if df.empty:
        print("No data for this subset.")
        return
    higher_is_better = not _is_time_metric(metric_name)

    summary_df = (
        df.groupby("classifier")
          .agg(mean=("value", "mean"),
               std=("value", "std"),
               median=("value", "median"),
               iqr=("value", lambda x: float(np.subtract(*np.percentile(x, [75, 25]))) if len(x) else np.nan),
               n_folds=("value", "count"))
          .reset_index()
          .sort_values("mean", ascending=not higher_is_better)
          .reset_index(drop=True)
    )

    print("=" * 80)
    print(f"Metric: {metric_name} | Data: CV folds (development)")
    print("-" * 80)
    print("Per-classifier summary:")
    display(summary_df[["classifier", "n_folds", "mean", "std", "median", "iqr"]].round(6))

    if summary_df["n_folds"].sum() < 3 or summary_df["classifier"].nunique() < 2:
        print("Not enough data for statistical comparison "
              "(need >= 2 classifiers and >= 3 total fold scores).")
        return

    matrix, folds, classifiers = _build_block_matrix(df, _CLASSIFIER_ORDER)
    if matrix is None:
        print("Block matrix could not be built (unbalanced design or missing folds).")
        print("Friedman / Wilcoxon require a complete fold-by-classifier matrix.")
        return

    print(f"\nBlock matrix shape: n_folds={matrix.shape[0]}, n_classifiers={matrix.shape[1]}")
    print(f"Classifiers in order: {classifiers}")

    const_cols = int(np.sum(np.isclose(matrix.std(axis=0), 0.0)))
    if const_cols:
        print(f"Note: {const_cols} classifier(s) have identical scores across all folds "
              "(ceiling/degenerate); paired differences involving them are exactly zero.")

    fried = _friedman_test(matrix)
    if fried is None or "error" in (fried or {}):
        err = (fried or {}).get("error", "unknown")
        print(f"Friedman test could not be computed: {err}")
        print("(This typically happens under a perfect ceiling, where every classifier "
              "scores identically across all folds — there is simply nothing to rank.)")
        return

    print("\nFriedman test results:")
    display(pd.DataFrame([{
        "chi2": fried["stat"],
        "df": fried["n_groups"] - 1,
        "p_value": fried["p"],
        "kendall_w": fried["kendall_w"],
        "n_blocks (folds)": fried["n_blocks"],
        "n_groups (classifiers)": fried["n_groups"],
    }]).round(6))

    p = fried["p"]
    if p < alpha:
        print(f"Friedman p = {p:.4g} < alpha = {alpha:.3f} -> at least one classifier differs.")
        do_posthoc = True
    else:
        print(f"Friedman p = {p:.4g} >= alpha = {alpha:.3f} -> no overall difference detected "
              "(at n=4 / under ceiling effects this is likely power-limited; see caveats).")
        do_posthoc = False

    pairwise_df = pd.DataFrame()
    if do_posthoc:
        print("\nPairwise Wilcoxon signed-rank (paired per-fold), Holm-Bonferroni adjusted:")
        pairwise_df = _wilcoxon_pairwise(matrix, classifiers, alpha=alpha)
        if pairwise_df.empty:
            print("No pairwise post-hoc results.")
        else:
            disp = pairwise_df.copy()
            for c in ("mean_diff", "ci_low", "ci_high", "p_value_adj"):
                disp[c] = disp[c].astype(float).round(6)
            display(disp)
            sig_pairs = pairwise_df.loc[pairwise_df["significant"]]
            if sig_pairs.empty:
                print("No classifier pairs are significantly different at the adjusted level.")
            else:
                print("\nSignificant pairwise differences:")
                for _, row in sig_pairs.iterrows():
                    print(f"  {row['classifier_A']} vs {row['classifier_B']}: "
                          f"mean_diff = {row['mean_diff']:.4g}, p_adj = {row['p_value_adj']:.4g}")
            print("\nPairwise confidence intervals (bootstrap):")
            _plot_ci_pairs(pairwise_df)

    summary_text = _build_text_summary(
        summary_df=summary_df, pairwise_df=pairwise_df,
        metric_name=metric_name, alpha=alpha,
        higher_is_better=higher_is_better, friedman_p=p,
    )
    print("\nSummary:")
    print(summary_text)

def run_classifier_stats(metric_name, alpha=0.05):
    """Entry point: run the full Friedman+Wilcoxon analysis for one metric on the CV folds."""
    if metric_name not in _METRICS:
        print(f"Metric '{metric_name}' not available. Choose from: {_METRICS}")
        return
    df_long = _build_long_df(metric_name)
    if df_long.empty:
        print(f"No data found for metric '{metric_name}'.")
        return
    print(f"Friedman + Wilcoxon classifier comparison | metric = '{metric_name}' "
          f"| data = CV folds (development)")
    print(f"Classifiers: {df_long['classifier'].nunique()} | "
          f"folds: {df_long['fold'].nunique()} | rows: {len(df_long)}\n")
    _analyze_one_subset(df_long, metric_name, alpha)

# --- Interactive widget (falls back to a direct run if ipywidgets is absent) ---
try:
    import ipywidgets as widgets
    _HAS_WIDGETS = True
except Exception:
    _HAS_WIDGETS = False

if _HAS_WIDGETS:
    metric_dropdown = widgets.Dropdown(
        options=_METRICS, value=_METRICS[0],
        description="Metric:", layout=widgets.Layout(width="320px"),
    )
    alpha_float = widgets.BoundedFloatText(
        value=0.05, min=0.0001, max=0.5, step=0.005,
        description="alpha:", layout=widgets.Layout(width="160px"),
    )
    run_button = widgets.Button(
        description="Run statistical analysis",
        button_style="primary", layout=widgets.Layout(width="250px"),
    )
    stat_output = widgets.Output()

    def _on_run_button_clicked(_):
        with stat_output:
            clear_output(wait=True)
            run_classifier_stats(metric_dropdown.value, float(alpha_float.value))

    run_button.on_click(_on_run_button_clicked)

    display(widgets.VBox([
        widgets.HTML("<b>Classifier comparison — Friedman + pairwise Wilcoxon (CV folds)</b>"),
        widgets.HBox([metric_dropdown, alpha_float]),
        run_button,
        stat_output,
    ]))
else:
    print("ipywidgets is not installed in this kernel — running non-interactively.")
    print("For the interactive metric/alpha selector: pip install ipywidgets")
    print(f"You can also call run_classifier_stats('<metric>', alpha) directly. "
          f"Available metrics: {_METRICS}\n")
    run_classifier_stats(_METRICS[0], 0.05)